In [1]:
# Homework 5 Monte Carlo 


# Problem formulation 
import numpy as np
import scipy

Sample_Num = 20000

Length = 100 # in
Displacement_Tolerance = 2.2535 # in

# minimize:
Area = lambda x: x[0] * x[1] # in ^ 2
# X[0] = Width 
# X[1] = Thickness
# X[2] = Random Yield Strength
# X[3] = Young's Modulus 
# X[4] = Horizontal Load
# X[5] = Vertical Load

Stress_Criteria = lambda x: x[2] - ((600/(x[0] * x[1] ** 2)) * x[5] + (600/(x[0] ** 2 * x[1])) * x[4])
Displacement_Criteria = lambda x: Displacement_Tolerance - (4 * (Length ** 3) / (x[3] * x[0] * x[1])) * np.sqrt((x[5]/(x[1]**2)**2+(x[4]/(x[0]**2))**2))


def Monte_Carlo_Test_Displacement(x):
    Width, Thickness = x
    Failure_Count_Displacment = 0
    
    for i in range(Sample_Num):
            Random_Yield_Strength = np.random.normal(40000,2000)
            Youngs_Modulus = np.random.normal(29e6,1.5e6)
            Horizontal_Load = np.random.normal(500,100)
            Vertical_Load = np.random.normal(1000,100)
            
            x = np.array([Width,Thickness,Random_Yield_Strength,Youngs_Modulus,Horizontal_Load,Vertical_Load])
            
            Displacement_Test = Displacement_Criteria(x)
            
            if Displacement_Test < 0:
                Failure_Count_Displacment += 1

    if Failure_Count_Displacment == 0:
        return 10 # Pass
    elif Failure_Count_Displacment == Sample_Num:
        return -10 # Fail
    else:
        Failure_Probability_Displacement = (Failure_Count_Displacment) / Sample_Num
        
        # norminv python equivalent
        Beta_Displacement = scipy.stats.norm.ppf(1 - Failure_Probability_Displacement)
    
            
        return Beta_Displacement

def Monte_Carlo_Test_Stress(x):
    Width, Thickness = x
    Failure_Count_Stress = 0

    for i in range(Sample_Num):
        Random_Yield_Strength = np.random.normal(40000,2000)
        Youngs_Modulus = np.random.normal(29e6,1.5e6)
        Horizontal_Load = np.random.normal(500,100)
        Vertical_Load = np.random.normal(1000,100)

        x = np.array([Width,Thickness,Random_Yield_Strength,Youngs_Modulus,Horizontal_Load,Vertical_Load])
        
        Stress_Test = Stress_Criteria(x)
        if Stress_Test < 0:
            Failure_Count_Stress += 1

    if Failure_Count_Stress == 0:
        return 10 # Pass
    elif Failure_Count_Stress == Sample_Num:
        return -10 # Fail
    else:
        Failure_Probability_Stress = (Failure_Count_Stress) / Sample_Num
    
        # norminv python equivalent
        Beta_Stress = scipy.stats.norm.ppf(1 - Failure_Probability_Stress)
        
        return Beta_Stress


Initial_Guess = np.array([2.7,4])
Constraints = (
    scipy.optimize.NonlinearConstraint(Monte_Carlo_Test_Displacement, 3, np.inf),
    scipy.optimize.NonlinearConstraint(Monte_Carlo_Test_Stress, 3, np.inf)
)
Bounds = (
    np.array([1, 4]),
    np.array([1, 4])
)

Monte_Carlo_Results = scipy.optimize.minimize(Area, Initial_Guess, constraints = Constraints, bounds = Bounds, method='trust-constr')

print("Width: ", Monte_Carlo_Results.x[0])
print("Thickness: ", Monte_Carlo_Results.x[1])
print("Stress Constraint 1: ", Monte_Carlo_Test_Stress(Monte_Carlo_Results.x))
print("Stress Constraint 2: ", Monte_Carlo_Test_Stress(Monte_Carlo_Results.x))
print("Stress Constraint 3: ", Monte_Carlo_Test_Stress(Monte_Carlo_Results.x))
print("Stress Constraint 4: ", Monte_Carlo_Test_Stress(Monte_Carlo_Results.x))
print("Stress Constraint 5: ", Monte_Carlo_Test_Stress(Monte_Carlo_Results.x))
print("Displacement Constraint: ", Monte_Carlo_Test_Displacement(Monte_Carlo_Results.x))
print("Area: ", Area(Monte_Carlo_Results.x))

C:\Users\natha\anaconda3\Lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


Width:  2.521442554636116
Thickness:  3.970141944158714
Stress Constraint 1:  3.6153000069246914
Stress Constraint 2:  3.719016485455709
Stress Constraint 3:  3.431614403623299
Stress Constraint 4:  3.719016485455709
Stress Constraint 5:  3.6153000069246914
Displacement Constraint:  10
Area:  10.010484845947543


In [2]:
# Homework 5 Form

# Problem formulation 
import numpy as np
import scipy

e_tol = 1e-6
num_iters = 1000

Length = 100 # in
Displacement_Tolerance = 2.2535 # in

# minimize:
Area = lambda x: x[0] * x[1] # in ^ 2
# X[0] = Width 
# X[1] = Thickness
# X[2] = Random Yield Strength
# X[3] = Young's Modulus 
# X[4] = Horizontal Load
# X[5] = Vertical Load

Stress_Criteria = lambda x: x[2] - ((600/(x[0] * x[1] ** 2)) * x[5] + (600/(x[0] ** 2 * x[1])) * x[4])
Displacement_Criteria = lambda x: Displacement_Tolerance - (4 * (Length ** 3) / (x[3] * x[0] * x[1])) * np.sqrt((x[5]/(x[1]**2)**2+(x[4]/(x[0]**2))**2))

h = 0.001
h0 = np.array([h,0,0,0,0,0])
h1 = np.array([0,h,0,0,0,0])
h2 = np.array([0,0,h,0,0,0])
h3 = np.array([0,0,0,h,0,0])
h4 = np.array([0,0,0,0,h,0])
h5 = np.array([0,0,0,0,0,h])

Stress_Criteria_Gradient = lambda x: np.array([
    (Stress_Criteria(x+h0)-Stress_Criteria(x-h0))/(h*2),
    (Stress_Criteria(x+h1)-Stress_Criteria(x-h1))/(h*2),
    (Stress_Criteria(x+h2)-Stress_Criteria(x-h2))/(h*2),
    (Stress_Criteria(x+h3)-Stress_Criteria(x-h3))/(h*2),
    (Stress_Criteria(x+h4)-Stress_Criteria(x-h4))/(h*2),
    (Stress_Criteria(x+h5)-Stress_Criteria(x-h5))/(h*2)])

Displacement_Criteria_Gradient = lambda x: np.array([
    (Displacement_Criteria(x+h0)-Displacement_Criteria(x-h0))/(h*2),
    (Displacement_Criteria(x+h1)-Displacement_Criteria(x-h1))/(h*2),
    (Displacement_Criteria(x+h2)-Displacement_Criteria(x-h2))/(h*2),
    (Displacement_Criteria(x+h3)-Displacement_Criteria(x-h3))/(h*2),
    (Displacement_Criteria(x+h4)-Displacement_Criteria(x-h4))/(h*2),
    (Displacement_Criteria(x+h5)-Displacement_Criteria(x-h5))/(h*2)])

def FORM_Test_Displacement(x):
    mu_x = np.array([x[0],x[1],40000,29e6,500,1000])
    std_x = np.array([0.00001,0.00001,2000,1.5e6,100,100])
    
    k = 1
    mu_g = Displacement_Criteria(mu_x)
    dg_dx = Displacement_Criteria_Gradient(mu_x)
    std_g = np.linalg.norm(dg_dx*std_x)
    beta_k = mu_g/std_g
    beta_k_next = beta_k
    dir_cos = -dg_dx*std_x/std_g
    test = 1
    while test > e_tol and k < num_iters:
        x_star = mu_x + (beta_k*std_x*dir_cos)
        u = (x_star-mu_x)/std_x
        std_g = np.linalg.norm(Displacement_Criteria_Gradient(x_star)*std_x)
        k = k+1
        beta_k_next = (Displacement_Criteria(x_star)-np.sum(Displacement_Criteria_Gradient(x_star)*std_x*u))/std_g
        dir_cos = (-Displacement_Criteria_Gradient(x_star)*std_x)/std_g
        test = abs(beta_k_next-beta_k)/beta_k_next
        beta_k = beta_k_next
    
    return beta_k_next

def FORM_Test_Stress(x):
    mu_x = np.array([x[0],x[1],40000,29e6,500,1000])
    std_x = np.array([0.00001,0.00001,2000,1.5e6,100,100])

    k = 1
    mu_g = Stress_Criteria(mu_x)
    dg_dx = Stress_Criteria_Gradient(mu_x)
    std_g = np.linalg.norm(dg_dx*std_x)
    beta_k = mu_g/std_g
    beta_k_next = beta_k
    dir_cos = -dg_dx*std_x/std_g
    test = 1
    while test > e_tol and k < num_iters:
        x_star = mu_x + (beta_k*std_x*dir_cos)
        u = (x_star-mu_x)/std_x
        std_g = np.linalg.norm(Stress_Criteria_Gradient(x_star)*std_x)
        k = k+1
        beta_k_next = (Stress_Criteria(x_star)-np.sum(Stress_Criteria_Gradient(x_star)*std_x*u))/std_g
        dir_cos = (-Stress_Criteria_Gradient(x_star)*std_x)/std_g
        test = abs(beta_k_next-beta_k)/beta_k_next
        beta_k = beta_k_next
        
    return beta_k_next

print(FORM_Test_Displacement([2.7,4]))


Initial_Guess = np.array([3,4])
Constraints = (
    scipy.optimize.NonlinearConstraint(FORM_Test_Displacement, 3, np.inf),
    scipy.optimize.NonlinearConstraint(FORM_Test_Stress, 3, np.inf)
)
Bounds = (
    np.array([1, 4]),
    np.array([1, 4])
)

FORM_Results = scipy.optimize.minimize(Area, Initial_Guess, constraints = Constraints, bounds = Bounds, method='trust-constr')

print("Width: ", FORM_Results.x[0])
print("Thickness: ", FORM_Results.x[1])
print("Stress Constraint: ", FORM_Test_Stress(FORM_Results.x))
print("Displacement Constraint: ", FORM_Test_Displacement(FORM_Results.x))
print("Area: ", Area(FORM_Results.x))


6.545719414293101
Width:  2.445992147768851
Thickness:  3.8921822313663177
Stress Constraint:  3.0000000032525858
Displacement Constraint:  3.8789321653683544
Area:  9.520247175607459


[1. 1. 1.]
